In [1]:
import os, sys, pathlib

cwd = pathlib.Path().resolve()          # notebook working dir
# climb until we find a folder that contains a `src` directory
proj = cwd
while proj != proj.parent and not (proj / "src").exists():
    proj = proj.parent

if (proj / "src").exists():
    sys.path.insert(0, str(proj))     # add project root (parent of `src`)
else:
    # fallback: ensure parent of cwd is on sys.path
    sys.path.insert(0, str(cwd.parent))

import logging
import numpy as np
import pandas as pd

from src.core.database import get_engine

In [2]:


def extract_from_bronze(table_name: str) -> pd.DataFrame:
    engine = get_engine("bronze")
    try:
        return pd.read_sql(f"SELECT * FROM {table_name}", engine)
    except Exception as e:
        raise RuntimeError(f"Failed to extract from bronze table {table_name}") from e

In [3]:
df= extract_from_bronze("crm_customers_info")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18494 entries, 0 to 18493
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   raw_row              18494 non-null  object
 1   cst_id               18490 non-null  object
 2   cst_key              18494 non-null  object
 3   cst_firstname        18486 non-null  object
 4   cst_lastname         18487 non-null  object
 5   cst_marital_status   18487 non-null  object
 6   cst_gndr             13916 non-null  object
 7   cst_create_date_raw  18490 non-null  object
dtypes: object(8)
memory usage: 1.1+ MB


In [4]:
#! Define schema for data types
schema_customer ={
    "cst_id"              : "string",
    "cst_key"             : "string",
    "cst_firstname"       : "string",
    "cst_lastname"        : "string",
    "cst_marital_status"  : "string",
    "cst_gndr"            : "string",
    "cst_create_date_raw" : "datetime64[ns]"
    }

#! combined normalization function for all string columns in the dataframe
def normalize_data(df: pd.DataFrame) -> pd.DataFrame:
    str_cols = df.select_dtypes(include="string").columns
    for col in str_cols:
        logging.info(f"Normalizing nulls in column: {col}")
        df[col] = (
            df[col]
            .str.strip()
            .replace(
                ["", "NULL", "null", "None", "none", "nan", "NaN"], 
                pd.NA
                )
        )
    df.drop("raw_row",axis=1,inplace=True)
    df[["cst_firstname","cst_lastname"]] = df[["cst_firstname","cst_lastname"]].apply(lambda x: x.str.strip().str.title())
    return df

#! high level schema enforcement function 
def enforce_schema(df: pd.DataFrame, schema: dict) -> pd.DataFrame:
    for column, dtype in schema.items():

        if column not in df.columns:
            #! log warning and skip missing columns
            logging.warning(f"[SCHEMA WARNING] Column missing: {column}")
            continue
        if dtype in ("Int64", "int64", "float64"):
            df[column] = pd.to_numeric(df[column], errors="coerce")
            if dtype == "Int64":
                df[column] = df[column].astype("Int64")
        elif dtype.startswith("datetime"):
            df[column] = pd.to_datetime(df[column], errors="coerce")
        elif dtype == "boolean":
            df[column] = df[column].astype("boolean")
        elif dtype == "string":
            df[column] = df[column].astype("string")
        else:
            # fallback (rare cases)
            df[column] = df[column].astype(dtype)

    return df

In [5]:
def data_quality_checks(df: pd.DataFrame):  
    PRIMARY_KEY = ["cst_id"]
    dup_mask = df.duplicated(subset=PRIMARY_KEY, keep=False)
    dup_rows = df[dup_mask]
    if not dup_rows.empty:
        dup_summary = (
        dup_rows
        .groupby(PRIMARY_KEY)
        .size()
        .reset_index(name="occurrences")
        )
        for _, row in dup_summary.iterrows():
            logging.info(
                f"[DUPLICATE FOUND] {PRIMARY_KEY[0]}={row['cst_id']} "
                f"→ occurrences={row['occurrences']}"
            )
    else:
        logging.info("No duplicates found")
    


In [6]:
def standardize_data(df: pd.DataFrame) -> pd.DataFrame:

    if df.empty:
        return df
    #! Gender Standardization
    df["cst_gndr"] = (
        df["cst_gndr"]
        .str.lower()
        .map({
            "m": "Male",
            "f": "Female",
        })
    )

    #!Marital Status Standardization
    df["cst_marital_status"] = (
        df["cst_marital_status"]
        .str.lower()
        .map({
            "s": "Single",
            "m": "Married",
        })
    )

    df[["cst_gndr","cst_marital_status"]] = df[["cst_gndr","cst_marital_status"]].fillna("Unknown")

    return df

In [7]:
#! clean function for deleting duplicate records 
def deduplicate(df: pd.DataFrame):
    PRIMARY_KEY = ["cst_id"]
    TIMESTAMP_COL = "cst_create_date_raw"

    if df.empty:
        logging.info("[DEDUP] Empty DataFrame.")
        return df, pd.DataFrame()
    #!Sort so latest loaded_at comes first
    df_sorted = df.sort_values(
        by=TIMESTAMP_COL,
        ascending=False
    )
    #!Identify duplicates (keep latest)
    dup_mask = df_sorted.duplicated(
        subset=PRIMARY_KEY,
        keep="first"
    )
    #! Separate kept vs deleted rows
    kept_rows = df_sorted[~dup_mask].copy()
    deleted_rows = df_sorted[dup_mask].copy()
    if not deleted_rows.empty:
        logging.info(f"[DEDUP] Found {len(deleted_rows)} duplicate rows based on {PRIMARY_KEY}")
    

    #! Logging
    logging.info(f"[DEDUP] Total rows   : {len(df)}")
    logging.info(f"[DEDUP] Kept rows    : {len(kept_rows)}")
    logging.info(f"[DEDUP] Deleted rows : {len(deleted_rows)}")
    return kept_rows, deleted_rows

In [8]:
def run_customers_pipeline(table_name: str):
    df_customers = extract_from_bronze(table_name)
    df_customers = enforce_schema(df_customers, schema_customer) # object → string, datetime → datetime64, etc.
    df_customers = normalize_data(df_customers)           
    df_customers = standardize_data(df_customers) # standardize gender and marital status values  

    data_quality_checks(df_customers) # log any duplicates found into log file (but do not remove yet)
    df_customers, df_duplicates = deduplicate(df_customers) # remove duplicates based on cst_id, keep latest loaded_at record

In [9]:
df_customers = extract_from_bronze(table_name="crm_customers_info")
df_customers

,raw_row,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date_raw
0,"""{\""cst_id\"": \""11000\"", \""cst_key\"": \""AW0001...",11000,AW00011000,Jon,Yang,M,M,2025-10-06
1,"""{\""cst_id\"": \""11001\"", \""cst_key\"": \""AW0001...",11001,AW00011001,Eugene,Huang,S,M,2025-10-06
2,"""{\""cst_id\"": \""11002\"", \""cst_key\"": \""AW0001...",11002,AW00011002,Ruben,Torres,M,M,2025-10-06
3,"""{\""cst_id\"": \""11003\"", \""cst_key\"": \""AW0001...",11003,AW00011003,Christy,Zhu,S,F,2025-10-06
4,"""{\""cst_id\"": \""11004\"", \""cst_key\"": \""AW0001...",11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
...,...,...,...,...,...,...,...,...
18489,"""{\""cst_id\"": \""29482\"", \""cst_key\"": \""AW0002...",29482,AW00029482,Clayton,Zhang,M,None,2026-01-25
18490,"""{\""cst_id\"": \""29483\"", \""cst_key\"": \""AW0002...",29483,AW00029483,None,Navarro,None,None,2026-01-25
18491,"""{\""cst_id\"": \""29483\"", \""cst_key\"": \""AW0002...",29483,AW00029483,Marc,Navarro,M,None,2026-01-27
18492,"""{\""cst_id\"": NaN, \""cst_key\"": \""13451235\"", ...",None,13451235,None,None,None,None,None


In [40]:
df_customers = enforce_schema(df_customers, schema_customer)
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18494 entries, 0 to 18493
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   raw_row              18494 non-null  object        
 1   cst_id               18490 non-null  string        
 2   cst_key              18494 non-null  string        
 3   cst_firstname        18486 non-null  string        
 4   cst_lastname         18487 non-null  string        
 5   cst_marital_status   18487 non-null  string        
 6   cst_gndr             13916 non-null  string        
 7   cst_create_date_raw  18490 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(1), string(6)
memory usage: 1.1+ MB


In [41]:
df_customers = normalize_data(df_customers)
df_customers

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date_raw
0,11000,AW00011000,Jon,Yang,M,M,2025-10-06
1,11001,AW00011001,Eugene,Huang,S,M,2025-10-06
2,11002,AW00011002,Ruben,Torres,M,M,2025-10-06
3,11003,AW00011003,Christy,Zhu,S,F,2025-10-06
4,11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
...,...,...,...,...,...,...,...
18489,29482,AW00029482,Clayton,Zhang,M,<NA>,2026-01-25
18490,29483,AW00029483,<NA>,Navarro,<NA>,<NA>,2026-01-25
18491,29483,AW00029483,Marc,Navarro,M,<NA>,2026-01-27
18492,<NA>,13451235,<NA>,<NA>,<NA>,<NA>,NaT


In [42]:
df_customers = standardize_data(df_customers)
df_customers

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date_raw
0,11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
1,11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06
2,11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06
3,11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06
4,11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
...,...,...,...,...,...,...,...
18489,29482,AW00029482,Clayton,Zhang,Married,Unknown,2026-01-25
18490,29483,AW00029483,<NA>,Navarro,Unknown,Unknown,2026-01-25
18491,29483,AW00029483,Marc,Navarro,Married,Unknown,2026-01-27
18492,<NA>,13451235,<NA>,<NA>,Unknown,Unknown,NaT


In [43]:
def deduplicate(df: pd.DataFrame, primary_key: list, date_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
	if df.empty:
		logging.info("[DEDUP] Empty DataFrame.")
		return df, pd.DataFrame()
	df = df.sort_values([primary_key, date_col, "loaded_at"], ascending=[True, False, False])
	kept_rows = df.drop_duplicates(subset=primary_key, keep="first")
	deleted_rows = df[~df.index.isin(kept_rows.index)]
	logging.info(f"[DEDUP] Total rows   : {len(df)}")
	logging.info(f"[DEDUP] Kept rows    : {len(kept_rows)}")
	logging.info(f"[DEDUP] Deleted rows : {len(deleted_rows)}")
	return kept_rows, deleted_rows



In [ ]:
def deduplicate_latest_by_date(
    df: pd.DataFrame,
    primary_key: str,
    date_col: str,
    tie_breaker_col: str = None
) -> tuple[pd.DataFrame, pd.DataFrame]:

    if df.empty:
        logging.info("[DEDUP] Empty DataFrame.")
        return df, pd.DataFrame()

    df = df.copy()
    sort_cols = [primary_key, date_col]
    ascending_order = [True, False]    
    # Sort so latest records come first per primary key
    df_sorted = df.sort_values(by=sort_cols, ascending=ascending_order)
    # Keep latest per primary key
    kept_rows = df_sorted.drop_duplicates(subset=primary_key, keep="first")
    #! Identify deleted rows based on index difference
    deleted_rows = df_sorted[~df_sorted.index.isin(kept_rows.index)]
	#! logging into log file for debugging and monitoring how many duplicates were found and removed
    logging.info(f"[DEDUP] Total rows   : {len(df)}")
    logging.info(f"[DEDUP] Kept rows    : {len(kept_rows)}")
    logging.info(f"[DEDUP] Deleted rows : {len(deleted_rows)}")

    return kept_rows, deleted_rows

In [49]:
df_customers, df_duplicates = deduplicate_latest_by_date(
        df_customers,
        primary_key="cst_id",
        date_col="cst_create_date_raw"
    )

In [50]:
df_customers

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date_raw
0,11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
1,11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06
2,11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06
3,11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06
4,11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
...,...,...,...,...,...,...,...
18487,29480,AW00029480,Nina,Raji,Single,Unknown,2026-01-25
18488,29481,AW00029481,Ivan,Suri,Single,Unknown,2026-01-25
18489,29482,AW00029482,Clayton,Zhang,Married,Unknown,2026-01-25
18491,29483,AW00029483,Marc,Navarro,Married,Unknown,2026-01-27
